In [27]:
# ════════════════════════════════════════════════════════
# CELL 1 — Install + Import + Dataset Paths
# ════════════════════════════════════════════════════════

# Install first
!pip install -q open-clip-torch hnswlib

# Then import
import os, json
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import open_clip
import hnswlib

# 1. ⚠️ DEFINE YOUR KAGGLE DATASET PATHS ⚠️
CROPS_INPUT    = '/kaggle/input/datasets/saisandeepn/deep-fashion-crops'          # From Notebook 1
CAPTIONS_INPUT = '/kaggle/input/datasets/khsudhir/captions'                    # From Notebook 2
MODELS_INPUT   = '/kaggle/input/datasets/sudhirkh/deepfashion-models' # From Notebook 3

PROJ   = '/kaggle/working'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 2. CHOOSE YOUR SEED
# You trained 4 models in Notebook 3. We pick the first one to test here.
TEST_SEED = 546  

print(f"Device: {device}")
print(f"Testing Seed: {TEST_SEED}")

Device: cuda
Testing Seed: 546


In [28]:
# ════════════════════════════════════════════════════════
# CELL 2 — Shared embedding helpers
# ════════════════════════════════════════════════════════
def load_clip(finetuned: bool):
    """Load CLIP. If finetuned=True, loads our fine-tuned weights."""
    model, _, preprocess = open_clip.create_model_and_transforms(
        'ViT-B-32', pretrained='openai'
    )
    if finetuned:
        # >>> CRITICAL FIX: Load the exact model seed from Notebook 3 <<<
        ckpt_path = f'{MODELS_INPUT}/checkpoints/clip_ft_C_seed{TEST_SEED}_best.pt'
        
        state = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(state)
        print(f"Fine-tuned CLIP weights loaded (Seed {TEST_SEED}) ✓")
    else:
        print("Frozen (pretrained) CLIP loaded ✓")

    model = model.to(device).eval()
    tokenizer = open_clip.get_tokenizer('ViT-B-32')
    return model, preprocess, tokenizer

@torch.no_grad()
def image_embedding(model, preprocess, img_path: str) -> np.ndarray:
    """ϕV(x̂i) — CLIP vision encoder on a cropped image → (512,)"""
    img  = Image.open(img_path).convert('RGB')
    x    = preprocess(img).unsqueeze(0).to(device)
    emb  = model.encode_image(x)
    return F.normalize(emb, dim=-1).cpu().numpy().squeeze().astype('float32')

@torch.no_grad()
def text_embedding(model, tokenizer, caption: str) -> np.ndarray:
    """ϕT(ci) — CLIP text encoder on a BLIP-2 caption string → (512,)"""
    tokens = tokenizer([caption]).to(device)
    emb    = model.encode_text(tokens)
    return F.normalize(emb, dim=-1).cpu().numpy().squeeze().astype('float32')

def fused_embedding(img_emb: np.ndarray,
                    txt_emb: np.ndarray,
                    alpha: float) -> np.ndarray:
    """
    Equation (1) from PDF:
      vi = α·ϕV(x̂i) + (1-α)·ϕT(ci),  ||vi|| = 1
    """
    vi   = alpha * img_emb + (1 - alpha) * txt_emb
    norm = np.linalg.norm(vi)
    return (vi / norm).astype('float32') if norm > 1e-8 else vi

In [29]:
# ════════════════════════════════════════════════════════
# CELL 3 — Build HNSW index for ablation configs
# ════════════════════════════════════════════════════════
def build_index(config_name: str, finetuned: bool, alpha: float):
    print(f"\n{'='*50}")
    print(f"Building index: Config {config_name} | finetuned={finetuned} | α={alpha}")
    print(f"{'='*50}")

    # 1. Load project datasets from Kaggle Inputs
    df        = pd.read_csv(f'{CROPS_INPUT}/dataset_index.csv')
    crop_meta = json.load(open(f'{CROPS_INPUT}/crop_meta_gt.json'))
    
    # Safely load captions depending on how Kaggle unzipped the folder
    try:
        captions = json.load(open(f'{CAPTIONS_INPUT}/captions/captions.json'))
    except FileNotFoundError:
        captions = json.load(open(f'{CAPTIONS_INPUT}/captions.json'))

    # Index ONLY gallery images (the searchable database)
    gallery_df = df[df['split'] == 'gallery'].reset_index(drop=True)
    print(f"Gallery images to index: {len(gallery_df):,}")

    model, preprocess, tokenizer = load_clip(finetuned)

    DIM   = 512
    index = hnswlib.Index(space='cosine', dim=DIM)
    index.init_index(
        max_elements=len(gallery_df),
        ef_construction=200,   
        M=16                   
    )

    all_embeddings = np.zeros((len(gallery_df), DIM), dtype='float32')
    all_item_ids   = []
    all_paths      = [] 

    for i, row in tqdm(gallery_df.iterrows(), total=len(gallery_df), desc=f"Config {config_name}"):
        img_path = row['image_path']
        
        # >>> CRITICAL PATH TRANSLATION <<<
        old_crop_path = crop_meta.get(img_path)
        crop_path = old_crop_path.replace('/kaggle/working', CROPS_INPUT)
        
        caption   = captions.get(img_path, '')

        # Compute embedding based on config
        img_emb = image_embedding(model, preprocess, crop_path)

        if alpha < 1.0 and caption:
            txt_emb = text_embedding(model, tokenizer, caption)
            vi      = fused_embedding(img_emb, txt_emb, alpha)
        else:
            vi = img_emb

        all_embeddings[i] = vi
        all_item_ids.append(row['item_id'])
        all_paths.append(img_path)

    # Add all vectors to index
    index.add_items(all_embeddings, list(range(len(gallery_df))))
    index.set_ef(100)

    # Save index + consolidated metadata into config.json
    out_dir = f'{PROJ}/index/config_{config_name}'
    os.makedirs(out_dir, exist_ok=True)

    index.save_index(f'{out_dir}/hnsw.bin')
    np.save(f'{out_dir}/embeddings.npy', all_embeddings)
    
    with open(f'{out_dir}/config.json', 'w') as f:
        json.dump({
            'finetuned': finetuned, 
            'alpha': alpha,
            'item_ids': all_item_ids,
            'paths': all_paths
        }, f)

    print(f"Config {config_name} index and config.json saved → {out_dir}")
    return out_dir

# Build all three indexes as required by the ablation study
build_index('A', finetuned=False, alpha=1.0)
build_index('B', finetuned=False, alpha=0.5)
build_index('C', finetuned=True,  alpha=0.5)


Building index: Config A | finetuned=False | α=1.0
Gallery images to index: 12,612


/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Frozen (pretrained) CLIP loaded ✓


Config A:   0%|          | 0/12612 [00:00<?, ?it/s]

Config A index and config.json saved → /kaggle/working/index/config_A

Building index: Config B | finetuned=False | α=0.5
Gallery images to index: 12,612


...

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Fine-tuned CLIP weights loaded (Seed 546) ✓


Config C:   0%|          | 0/12612 [00:00<?, ?it/s]

Config C index and config.json saved → /kaggle/working/index/config_C


'/kaggle/working/index/config_C'

In [30]:
# ════════════════════════════════════════════════════════
# CELL 4 — The three retrieval metrics (from PDF)
# ════════════════════════════════════════════════════════
import numpy as np

def recall_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
    """
    PDF: "Fraction of queries for which at least one relevant
    item is retrieved in the top-K results."
    Returns 1.0 or 0.0 per query.
    """
    return float(bool(set(retrieved_ids[:k]) & relevant_ids))


def ndcg_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
    """
    PDF: "Ranking metric that gives higher score when relevant items
    appear earlier in the top-K list (position-aware gain)."
    """
    dcg = sum(
        1.0 / np.log2(rank + 2)
        for rank, rid in enumerate(retrieved_ids[:k])
        if rid in relevant_ids
    )
    # Ideal DCG: all relevant items at top positions
    ideal_hits = min(len(relevant_ids), k)
    idcg = sum(1.0 / np.log2(rank + 2) for rank in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0


def ap_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
    """
    PDF: "Mean of per-query average precision up to K, rewarding
    systems that retrieve relevant items in better rank positions."
    This function computes per-query AP; mAP is mean over queries.
    """
    hits, precision_sum = 0, 0.0
    for rank, rid in enumerate(retrieved_ids[:k], start=1):
        if rid in relevant_ids:
            hits          += 1
            precision_sum += hits / rank
    denom = min(len(relevant_ids), k)
    return precision_sum / denom if denom > 0 else 0.0

In [31]:
# ════════════════════════════════════════════════════════
# CELL 5 — Evaluate one config on query split (PATH FIXED)
# ════════════════════════════════════════════════════════
import os
import json
import torch
import numpy as np
import pandas as pd
import hnswlib
from tqdm.auto import tqdm

def evaluate_config(config_name: str, finetuned: bool, Ks=(5, 10, 15)):
    # 1. Load datasets from your attached Input datasets
    df         = pd.read_csv(f'{CROPS_INPUT}/dataset_index.csv')
    crop_meta  = json.load(open(f'{CROPS_INPUT}/crop_meta_yolo.json'))
    query_df   = df[df['split'] == 'query'].reset_index(drop=True)
    gallery_df = df[df['split'] == 'gallery'].reset_index(drop=True)

    gt_map = gallery_df.groupby('item_id')['item_id'].apply(set).to_dict()

    # 2. Load the index from the exact folder Cell 3 just created
    idx_dir = f'{PROJ}/index/config_{config_name}'
    idx = hnswlib.Index(space='cosine', dim=512)
    idx.load_index(f'{idx_dir}/hnsw.bin')
    idx.set_ef(150)

    meta = json.load(open(f'{idx_dir}/config.json')) 
    gallery_item_ids = meta['item_ids']
    gallery_paths    = meta['paths']

    # 3. Load the model 
    # (Cell 2's load_clip already pulls the exact seed model from your zip file!)
    model, preprocess, _ = load_clip(finetuned) 

    scores = {k: {'R': [], 'NDCG': [], 'AP': []} for k in Ks}
    max_k  = max(Ks)

    for _, row in tqdm(query_df.iterrows(), total=len(query_df),
                       desc=f"Eval Config {config_name} (Seed {TEST_SEED})"):
        
        # >>> CRITICAL PATH TRANSLATION <<<
        old_crop_path = crop_meta.get(row['image_path'])
        if not old_crop_path:
            continue
            
        # Hot-swap the path to point to your attached dataset
        crop_path = old_crop_path.replace('/kaggle/working', CROPS_INPUT)
        
        if not os.path.exists(crop_path):
            continue

        q_emb = image_embedding(model, preprocess, crop_path)
        labels, _ = idx.knn_query(q_emb, k=max_k + 5)

        retrieved_ids = []
        for lbl in labels[0]:
            if gallery_paths[lbl] != row['image_path']:
                # >>> NEW FIX: Only add the item if it's not already in the list <<<
                item_name = gallery_item_ids[lbl]
                if item_name not in retrieved_ids:
                    retrieved_ids.append(item_name)
            
            if len(retrieved_ids) == max_k:
                break

        relevant_ids = gt_map.get(row['item_id'], set())

        for k in Ks:
            scores[k]['R'].append(recall_at_k(retrieved_ids, relevant_ids, k))
            scores[k]['NDCG'].append(ndcg_at_k(retrieved_ids, relevant_ids, k))
            scores[k]['AP'].append(ap_at_k(retrieved_ids, relevant_ids, k))

    summary = {}
    for k in Ks:
        summary[f'Recall@{k}'] = float(np.mean(scores[k]['R']))
        summary[f'NDCG@{k}']   = float(np.mean(scores[k]['NDCG']))
        summary[f'mAP@{k}']    = float(np.mean(scores[k]['AP']))

    return summary

In [32]:
# ════════════════════════════════════════════════════════
# CELL 6 — Run full ablation for the chosen seed (CRASH-PROOF)
# ════════════════════════════════════════════════════════
import torch, numpy as np, json, os

# The three configurations we need to test
CONFIGS = [
    ('A', False),  # Baseline: Vision only, no fine-tuning
    ('B', False),  # Baseline: Vision + Text (50/50), no fine-tuning
    ('C', True),   # Your Model: Vision + Text (50/50), FINE-TUNED
]

all_results = {}

print(f"🌟 STARTING FINAL EVALUATION FOR SEED: {TEST_SEED} 🌟")

for cfg_name, finetuned in CONFIGS:
    key = f'Config_{cfg_name}'
    print(f"\n{'━'*50}")
    print(f"Evaluating {key}...")
    
    # Run the evaluation! (Cell 5 handles the paths, Cell 2 handles the model)
    result = evaluate_config(cfg_name, finetuned)
    all_results[key] = result
    
    # Flush GPU memory between configs so Kaggle doesn't crash
    torch.cuda.empty_cache()

# Save the results
os.makedirs(f'{PROJ}/results', exist_ok=True)
save_path = f'{PROJ}/results/ablation_seed{TEST_SEED}.json'

with open(save_path, 'w') as f:
    json.dump(all_results, f, indent=2)

# ─── PRINT THE FINAL SCOREBOARD ─────────────────────────
print(f"\n\n🏆 FINAL SCOREBOARD (SEED {TEST_SEED}) 🏆")
print("━"*50)
for cfg, scores in all_results.items():
    print(f"{cfg}:")
    print(f"  Recall@5: {scores['Recall@5']:.4f}  |  mAP@5: {scores['mAP@5']:.4f}")
    print(f"  Recall@15:{scores['Recall@15']:.4f}  |  mAP@15:{scores['mAP@15']:.4f}")
    print("━"*50)

print(f"Results saved safely to: {save_path}")

🌟 STARTING FINAL EVALUATION FOR SEED: 546 🌟

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Evaluating Config_A...


/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Frozen (pretrained) CLIP loaded ✓


Eval Config A (Seed 546):   0%|          | 0/14218 [00:00<?, ?it/s]


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Evaluating Config_B...


...


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Evaluating Config_C...


/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Fine-tuned CLIP weights loaded (Seed 546) ✓


Eval Config C (Seed 546):   0%|          | 0/14218 [00:00<?, ?it/s]



🏆 FINAL SCOREBOARD (SEED 546) 🏆
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Config_A:
  Recall@5: 0.2143  |  mAP@5: 0.1484
  Recall@15:0.3091  |  mAP@15:0.1592
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Config_B:
  Recall@5: 0.2655  |  mAP@5: 0.1647
  Recall@15:0.3855  |  mAP@15:0.1782
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Config_C:
  Recall@5: 0.8088  |  mAP@5: 0.6773
  Recall@15:0.8870  |  mAP@15:0.6867
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Results saved safely to: /kaggle/working/results/ablation_seed546.json


In [33]:
# ════════════════════════════════════════════════════════
# CELL 7 — Results table: mean ± std across seeds (FIXED)
# ════════════════════════════════════════════════════════
import pandas as pd, numpy as np, json, os

# Your team's roll numbers
SEEDS   = [546, 574, 580, 598]
PROJ    = '/kaggle/working'
Ks      = [5, 10, 15]

# Combine all individual seed results into one master dictionary
results = {}
seeds_found = []

for s in SEEDS:
    filepath = f'{PROJ}/results/ablation_seed{s}.json'
    if os.path.exists(filepath):
        seed_data = json.load(open(filepath))
        # Format the keys to match what the table generator expects
        for cfg_key, metrics in seed_data.items(): 
            cfg_letter = cfg_key.split('_')[1] # extracts 'A', 'B', or 'C'
            results[f'config_{cfg_letter}_seed{s}'] = metrics
        seeds_found.append(s)

if not seeds_found:
    print("❌ No evaluation results found! Did you run Cell 6?")
else:
    print(f"📊 Generating report for seeds: {seeds_found}\n")

    rows = []
    for cfg in ['A', 'B', 'C']:
        for metric in ([f'Recall@{k}' for k in Ks] +
                       [f'NDCG@{k}'   for k in Ks] +
                       [f'mAP@{k}'    for k in Ks]):
            
            vals = [
                results[f'config_{cfg}_seed{s}'][metric]
                for s in seeds_found
                if f'config_{cfg}_seed{s}' in results
            ]
            
            if vals:
                mean_val = np.mean(vals)
                # If only 1 seed is found, standard deviation is 0
                std_val  = np.std(vals) if len(vals) > 1 else 0.0
                
                rows.append({
                    'Config': cfg,
                    'Metric': metric,
                    'Mean':   round(mean_val, 4),
                    'Std':    round(std_val,  4),
                    'Display': f"{mean_val:.4f} ± {std_val:.4f}"
                })

    full_df = pd.DataFrame(rows)
    pivot   = full_df.pivot_table(
        index='Metric', columns='Config', values='Display', aggfunc='first'
    )
    
    # Sort index properly so Recall@5 comes before Recall@10
    ordered_metrics = []
    for metric_base in ['Recall', 'NDCG', 'mAP']:
         for k in Ks:
             ordered_metrics.append(f'{metric_base}@{k}')
    pivot = pivot.reindex(ordered_metrics)

    print(pivot.to_string())

    full_df.to_csv(f'{PROJ}/results/ablation_summary.csv', index=False)
    pivot.to_csv(f'{PROJ}/results/ablation_pivot.csv')
    print(f"\n✅ Saved summary tables to {PROJ}/results/")

📊 Generating report for seeds: [546, 574, 580, 598]

Config                   A                B                C
Metric                                                      
Recall@5   0.2144 ± 0.0001  0.2656 ± 0.0002  0.8037 ± 0.0054
Recall@10  0.2727 ± 0.0001  0.3400 ± 0.0001  0.8612 ± 0.0042
Recall@15  0.3092 ± 0.0001  0.3855 ± 0.0000  0.8842 ± 0.0044
NDCG@5     0.1649 ± 0.0000  0.1898 ± 0.0001  0.7064 ± 0.0057
NDCG@10    0.1838 ± 0.0001  0.2138 ± 0.0001  0.7251 ± 0.0053
NDCG@15    0.1935 ± 0.0001  0.2258 ± 0.0000  0.7312 ± 0.0053
mAP@5      0.1485 ± 0.0000  0.1648 ± 0.0001  0.6738 ± 0.0058
mAP@10     0.1564 ± 0.0000  0.1746 ± 0.0001  0.6816 ± 0.0056
mAP@15     0.1592 ± 0.0000  0.1782 ± 0.0001  0.6834 ± 0.0056

✅ Saved summary tables to /kaggle/working/results/


In [34]:
# ════════════════════════════════════════════════════════
# CELL 8 — Batch eval script (Deliverable 3 from PDF)
# ════════════════════════════════════════════════════════
import os

# We use an f-string here so we can inject your exact Kaggle paths for testing,
# but the script is set up to let the TA override them via command line.
batch_eval_script = f'''"""
batch_eval.py — Deliverable 3
Usage:
  python batch_eval.py --query_folder /path/to/query_images --config C --seed {TEST_SEED}
"""
import argparse, json, os, torch, numpy as np, pandas as pd
import torch.nn.functional as F, open_clip, hnswlib
from PIL import Image
from ultralytics import YOLO
from tqdm import tqdm

def recall_at_k(ret, rel, k):
    return float(bool(set(ret[:k]) & rel))

def ndcg_at_k(ret, rel, k):
    dcg  = sum(1/np.log2(r+2) for r, x in enumerate(ret[:k]) if x in rel)
    idcg = sum(1/np.log2(r+2) for r in range(min(len(rel),k)))
    return dcg/idcg if idcg else 0.0

def ap_at_k(ret, rel, k):
    h, s = 0, 0.0
    for r, x in enumerate(ret[:k], 1):
        if x in rel:
            h += 1; s += h/r
    d = min(len(rel), k)
    return s/d if d else 0.0

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--query_folder', required=True)
    parser.add_argument('--config', default='C', choices=['A','B','C'])
    parser.add_argument('--seed', default={TEST_SEED}, type=int)
    
    # Defaulting to your Kaggle paths, but TAs can override this when they test it
    parser.add_argument('--models_dir',  default='{MODELS_INPUT}/checkpoints')
    parser.add_argument('--index_dir',   default='{PROJ}/index')
    parser.add_argument('--dataset_csv', default='{CROPS_INPUT}/dataset_index.csv')
    args = parser.parse_args()

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # 1. Load CLIP Model
    finetuned = (args.config == 'C')
    model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
    
    if finetuned:
        model_path = f"{{args.models_dir}}/clip_ft_{{args.config}}_seed{{args.seed}}_best.pt"
        model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device).eval()

    # 2. Load YOLO (will download yolov8n.pt automatically if not found)
    yolo = YOLO('yolov8n.pt')

    # 3. Load HNSW Index & Metadata (Using our clean folder structure!)
    idx_folder = f"{{args.index_dir}}/config_{{args.config}}"
    
    idx = hnswlib.Index(space='cosine', dim=512)
    idx.load_index(f"{{idx_folder}}/hnsw.bin")
    idx.set_ef(150)
    
    gallery_item_ids = json.load(open(f"{{idx_folder}}/config.json"))['item_ids']

    df = pd.read_csv(args.dataset_csv)
    gallery_df = df[df['split']=='gallery']
    gt_map = gallery_df.groupby('item_id')['item_id'].apply(set).to_dict()

    Ks = [5, 10, 15]
    scores = {{k: {{'R':[],'NDCG':[],'AP':[]}} for k in Ks}}

    query_files = [f for f in os.listdir(args.query_folder) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    print(f"Found {{len(query_files)}} query images")

    for fname in tqdm(query_files, desc="Batch eval"):
        path = os.path.join(args.query_folder, fname)
        try:
            img     = Image.open(path).convert('RGB')
            results = yolo(img, verbose=False)
            boxes   = results[0].boxes
            best_box, best_area = None, 0
            for i, cls in enumerate(boxes.cls.cpu().numpy()):
                if int(cls) == 0:  # 0 is the 'person' class in YOLO
                    x1,y1,x2,y2 = boxes.xyxy[i].cpu().numpy()
                    area = (x2-x1)*(y2-y1)
                    if area > best_area:
                        best_area = area
                        best_box  = (int(x1),int(y1),int(x2),int(y2))
            crop = img.crop(best_box) if best_box else img

            x   = preprocess(crop).unsqueeze(0).to(device)
            with torch.no_grad():
                emb = F.normalize(model.encode_image(x), dim=-1).cpu().numpy().squeeze()

            labels, _ = idx.knn_query(emb, k=max(Ks)+5)
            retrieved  = [gallery_item_ids[l] for l in labels[0]][:max(Ks)]

            item_id    = os.path.splitext(fname)[0]
            relevant   = gt_map.get(item_id, set())

            for k in Ks:
                scores[k]['R'].append(recall_at_k(retrieved, relevant, k))
                scores[k]['NDCG'].append(ndcg_at_k(retrieved, relevant, k))
                scores[k]['AP'].append(ap_at_k(retrieved, relevant, k))

        except Exception as e:
            print(f"  Error on {{fname}}: {{e}}")

    print("\\n" + "="*45)
    print(f"Config {{args.config}} (Seed {{args.seed}}) results:")
    print("="*45)
    for k in Ks:
        r = np.mean(scores[k]['R']); n = np.mean(scores[k]['NDCG']); m = np.mean(scores[k]['AP'])
        print(f"  K={{k:2d}} | Recall={{r:.4f}} | NDCG={{n:.4f}} | mAP={{m:.4f}}")

if __name__ == '__main__':
    main()
'''

with open(f'{PROJ}/batch_eval.py', 'w') as f:
    f.write(batch_eval_script)

print(f"✅ Batch eval script saved → {PROJ}/batch_eval.py")

✅ Batch eval script saved → /kaggle/working/batch_eval.py


In [35]:
# ════════════════════════════════════════════════════════
# CELL 9 — Export Final Deliverables (Clean Zip)
# ════════════════════════════════════════════════════════
import zipfile
import os

zip_name = '/kaggle/working/final_project_deliverables.zip'

print("Bundling final project files...")

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    
    # 1. Grab the Python script
    if os.path.exists(f'{PROJ}/batch_eval.py'):
        zf.write(f'{PROJ}/batch_eval.py', 'batch_eval.py')
        print("  ✓ Added: batch_eval.py")

    # 2. Grab the final score CSVs
    results_dir = f'{PROJ}/results'
    if os.path.exists(results_dir):
        for f in os.listdir(results_dir):
            if f.endswith('.csv'):
                zf.write(os.path.join(results_dir, f), f'results/{f}')
                print(f"  ✓ Added table: {f}")

    # 3. Grab the Search Index Databases (HNSW)
    index_dir = f'{PROJ}/index'
    if os.path.exists(index_dir):
        for root, dirs, files in os.walk(index_dir):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, PROJ)
                zf.write(file_path, arcname)
        print("  ✓ Added: Vector Search Databases (Config A, B, C)")

print(f"\n🎉 DONE! 🎉")
print(f"Created '{zip_name}' ({os.path.getsize(zip_name)/1e6:.1f} MB)")
print("Download this zip file from the Kaggle sidebar. You are officially finished!")

Bundling final project files...
  ✓ Added: batch_eval.py
  ✓ Added table: ablation_summary.csv
  ✓ Added table: ablation_pivot.csv
  ✓ Added: Vector Search Databases (Config A, B, C)

🎉 DONE! 🎉
Created '/kaggle/working/final_project_deliverables.zip' (147.1 MB)
Download this zip file from the Kaggle sidebar. You are officially finished!
